# Qwen3.5 4B Text Fine-Tuning for IntersectionQA

Adapted from Unsloth text fine-tuning patterns for IntersectionQA public JSONL exports. This notebook trains on the dataset's existing data model: each public row provides a model-facing `prompt`, canonical `answer`, `task_type`, split assignment, labels, diagnostics, and provenance.

This notebook is intentionally text-only. It does not load render artifacts, create placeholder images, or use a vision collator. Multimodal training can be added later as a separate notebook once text-only fine-tuning is stable.


## Dataset Input

Expected layout is the normal IntersectionQA export directory:

```text
intersectionqa_v0_1/
  train.jsonl
  validation.jsonl
  test_random.jsonl
  test_generator_heldout.jsonl
  test_object_pair_heldout.jsonl
  test_near_boundary.jsonl
  metadata.json
  source_manifest.json
```

Generate it locally with the text fine-tuning config. By default it writes to `data/intersectionqa_v0_1`:

```bash
rtk uv run python -m scripts.dataset.generate_dataset \
  --config configs/text_finetune.yaml
```

The config includes the current text-only MVP tasks plus implemented P1 prompt tasks: clearance buckets, tolerance-fit, counterfactual pairwise prompts, and ranking prompts.


## Installation


In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    !uv pip install -qqq         "torch==2.8.0" "triton>=3.3.0" bitsandbytes xformers==0.0.32.post2         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"         "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0 datasets
# causal_conv1d is supported only on torch==2.8.0.
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0


## Model


In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "unsloth/Qwen3.5-4B"

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit = False,  # Use True to reduce VRAM at some quality cost.
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## Load IntersectionQA Rows


In [ ]:
from pathlib import Path
import json, os, random
from collections import Counter

DATASET_DIR_CANDIDATES = [
    os.environ.get("INTERSECTIONQA_DATASET_DIR"),
    "data/intersectionqa_v0_1",
    "/content/intersectionqa_v0_1",
]
DATASET_DIR = next(
    (Path(value) for value in DATASET_DIR_CANDIDATES if value and Path(value).exists()),
    Path("data/intersectionqa_v0_1"),
)
print("DATASET_DIR:", DATASET_DIR)

TRAIN_SPLITS = ["train"]
EVAL_SPLITS = ["validation", "test_random", "test_near_boundary"]
TASK_TYPES = {
    "binary_interference",
    "relation_classification",
    "volume_bucket",
    "clearance_bucket",
    "tolerance_fit",
    "pairwise_interference",
    "ranking_normalized_intersection",
}

MAX_TRAIN_ROWS = 2_000  # Set to None for the full split.
MAX_EVAL_ROWS = 200
RANDOM_SEED = 3407

PUBLIC_ROW_REQUIRED_KEYS = {
    "id", "dataset_version", "split", "task_type", "prompt", "answer", "script",
    "geometry_ids", "source", "base_object_pair_id", "assembly_group_id", "labels",
    "diagnostics", "difficulty_tags", "label_policy", "hashes", "metadata",
}


def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            missing = PUBLIC_ROW_REQUIRED_KEYS - set(row)
            if missing:
                raise ValueError(f"{path}:{line_number}: missing public-row keys: {sorted(missing)}")
            if row["diagnostics"].get("label_status") != "ok":
                continue
            if row["task_type"] not in TASK_TYPES:
                continue
            yield row


def load_rows(splits, limit=None):
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f"DATASET_DIR does not exist: {DATASET_DIR}. Upload or mount an IntersectionQA export first."
        )
    rows = []
    for split in splits:
        path = DATASET_DIR / f"{split}.jsonl"
        if not path.exists():
            raise FileNotFoundError(f"Missing split file: {path}")
        rows.extend(read_jsonl(path))
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(rows)
    return rows[:limit] if limit else rows

train_rows = load_rows(TRAIN_SPLITS, MAX_TRAIN_ROWS)
eval_rows = load_rows(EVAL_SPLITS, MAX_EVAL_ROWS)

if not train_rows:
    raise ValueError("No training rows matched TASK_TYPES and TRAIN_SPLITS.")
if not eval_rows:
    raise ValueError("No evaluation rows matched TASK_TYPES and EVAL_SPLITS.")

print(f"train rows: {len(train_rows):,}")
print(f"eval rows: {len(eval_rows):,}")
print("train task distribution:", dict(Counter(row["task_type"] for row in train_rows)))
print("train answer distribution:", dict(Counter(row["answer"] for row in train_rows)))


## Convert Rows to Text Conversations


In [ ]:
from datasets import Dataset


def prompt_for_model(row):
    return row["prompt"].rstrip() + "\n\nReturn exactly the requested answer and nothing else."


def messages_for_row(row):
    return [
        {"role": "user", "content": prompt_for_model(row)},
        {"role": "assistant", "content": row["answer"]},
    ]


def convert_to_text_example(row):
    text = tokenizer.apply_chat_template(messages_for_row(row), tokenize=False)
    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token
    return {"text": text, "id": row["id"], "task_type": row["task_type"]}

converted_train_dataset = Dataset.from_list([convert_to_text_example(row) for row in train_rows])
converted_eval_dataset = Dataset.from_list([convert_to_text_example(row) for row in eval_rows])

sample_row = train_rows[0]
print(sample_row["id"], sample_row["split"], sample_row["task_type"], "answer=", sample_row["answer"])
print(prompt_for_model(sample_row)[:2_500])
print("\ntraining text preview:\n", converted_train_dataset[0]["text"][:2_500])


## Baseline Inference Before Fine-Tuning


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)


def generate_for_row(row, max_new_tokens=32):
    messages = [{"role": "user", "content": prompt_for_model(row)}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.1,
        min_p=0.1,
    )

print("Expected:", eval_rows[0]["answer"])
generate_for_row(eval_rows[0])


## Train


In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = converted_train_dataset,
    eval_dataset = converted_eval_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = RANDOM_SEED,
        output_dir = "outputs",
        report_to = "none",
        dataset_text_field = "text",
        max_length = 4096,
        packing = False,
    ),
)


In [ ]:

trainer_stats = trainer.train()
trainer_stats


## Inference After Fine-Tuning


In [ ]:
FastLanguageModel.for_inference(model)

for row in eval_rows[:5]:
    print("\nrow:", row["id"], "task:", row["task_type"], "expected:", row["answer"])
    generate_for_row(row)


## Save LoRA Adapters


In [ ]:

model.save_pretrained("intersectionqa_qwen_lora")
tokenizer.save_pretrained("intersectionqa_qwen_lora")
# model.push_to_hub("YOUR_USERNAME/intersectionqa_qwen_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_USERNAME/intersectionqa_qwen_lora", token="YOUR_HF_TOKEN")


## Optional Merged / GGUF Exports


In [ ]:

# Select only the export you need.

if False:
    model.save_pretrained_merged("intersectionqa_qwen_finetune", tokenizer)

if False:
    model.push_to_hub_merged(
        "YOUR_USERNAME/intersectionqa_qwen_finetune",
        tokenizer,
        token="YOUR_HF_TOKEN",
    )

if False:
    model.save_pretrained_gguf("intersectionqa_qwen_finetune", tokenizer, quantization_method="q8_0")

if False:
    model.push_to_hub_gguf(
        "YOUR_USERNAME/intersectionqa_qwen_finetune",
        tokenizer,
        quantization_method="q8_0",
        token="YOUR_HF_TOKEN",
    )
